# 02 · Modeling

**Goal:** fit several classifiers on 2022–2023 and leave 2024 for the comparison.

1. Load training data
2. GLM: a formula for the chance of a claim
3. Read the GLM coefficients
4. Five other classifiers, in plain language
5. Notes on modeling choices

In [ ]:
import importlib
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
import src.train as train_mod

importlib.reload(config)
importlib.reload(train_mod)
fit_glm, fit_all, MODEL_NOTES = train_mod.fit_glm, train_mod.fit_all, train_mod.MODEL_NOTES

In [ ]:
train = pd.read_csv(config.TRAIN_FILE)
print(train.shape, f"claim rate = {train[config.TARGET].mean():.2%}")

## 2. GLM

Why the **cloglog link with a log(exposure) offset**? If claims arrive as a Poisson process with rate λ, the probability of at least one claim over exposure *t* is 1 − exp(−λt). Taking cloglog of that gives log(λ) + log(t), so log(exposure) enters as an offset. TODO: explain this in your own words in the report.

In [ ]:
glm = fit_glm(train)
glm.summary()

## 3. Relativities

exp(β) is the multiplicative effect on the claim rate. Example: 1.25 means 25% higher risk than the reference level, all else equal.

In [ ]:
relativities = pd.DataFrame({
    "relativity": np.exp(glm.params),
    "ci_low": np.exp(glm.conf_int()[0]),
    "ci_high": np.exp(glm.conf_int()[1]),
    "p_value": glm.pvalues,
}).round(3)
relativities

**Interpretation:**

Vehicle value matters most. From the 25th to the 75th percentile of declared value (about 14,000), the claim rate is multiplied by about 1.16, holding age fixed. Vehicle age is next: an 18-year gap, the same kind of middle spread, multiplies the rate by about 0.82, so older vehicles claim less often. Driver age has a relativity of about 1.00 (p = 0.99). A 38-year gap from the 5th to the 95th percentile barely moves the rate.

The flat driver-age result is the surprise. Driver age and vehicle age move together in this portfolio (correlation 0.86), so once vehicle age is in the model the linear driver-age term has nothing left to explain. A straight line also cannot pick up a young-driver spike. All three factors together still leave most claims unexplained (Cox–Snell pseudo R² about 0.005).

## 4. Other classifiers

Each model below answers the same question as the GLM: the probability of at least one claim. The predictors are driver age, vehicle age, vehicle value, and exposure. Settings were fixed before looking at 2024.

- **Logistic regression.** One straight line for the log-odds of a claim. The inputs are scaled so vehicle value does not drown the ages.
- **Decision tree.** A few yes-or-no questions. Policies that take the same path share one claim rate. The tree stops after four questions.
- **Random forest.** Many trees, each trained on a random sample of policies. The probability is the average of the trees.
- **XGBoost.** Trees built in sequence. Each new tree corrects the policies the earlier trees got wrong.
- **LightGBM.** The same idea as XGBoost, with a different way of finding the splits.

In [ ]:
models = fit_all(train)
for name, note in MODEL_NOTES.items():
    print(f"{name}: {note}\n")

## 5. Modeling choices

Every model estimates the probability of at least one claim. About 14% of training policies have a claim, and that frequency is the base rate the probabilities should match. Oversampling or class weights would push the predictions toward a 50/50 problem, so they would no longer be the chance of a claim. A policy that was only in force for part of the year is handled in the GLM by a log(exposure) offset. The other models receive exposure as a column.

The tree settings were fixed before seeing 2024: the single tree is four questions deep, the forest has 200 trees, and XGBoost and LightGBM each have 300 trees with a learning rate of 0.05. There is no cross-validated search. Any later search has to stay inside 2022–2023.

The first GLM formula named `years_licensed` and `vehicle_type`. Those columns are not in the training file, so the fit failed until the formula used `driver_age`, `vehicle_age`, and `vehicle_value`. A linear driver-age term then added nothing once vehicle age was included. Rows with a missing predictor or zero exposure are left out of the GLM, because the offset is log(exposure).